# CLINICAL-CORE / TABULAR-CONN

## 1. Montar Google Drive

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 2. Instalar dependencias

In [3]:
!pip install -q lifelines pydantic pyyaml scikit-learn scipy

# TabPFN es opcional — si falla, Variante B usa fallback de proyección aleatoria

# !pip install -q tabpfn 2>/dev/null || echo "TabPFN no disponible — usando fallback"print("Dependencias instaladas")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 13.4 MB/s eta 0:00:00


## 3. Configurar rutas y copiar código a /content

In [4]:
# El código vive en Drive para versionado, pero lo copiamos a `/content` para imports limpios y para que cualquier cambio que hagas durante una sesión no contamine el código del Drive.

import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/data_tesis")

CODE_DIR_DRIVE = DRIVE_ROOT / "code"

DATA_DIR = DRIVE_ROOT / "clinicalsupplement"

RESULTS_DIR = DRIVE_ROOT / "results"

## 4. Inspeccionar y (opcionalmente) editar el experiment_config

In [5]:
import yaml
from pathlib import Path

# Define CODE_DIR_LOCAL and copy code as suggested in the text cell 3
CODE_DIR_LOCAL = Path("/content/code")
if not CODE_DIR_LOCAL.exists():
    shutil.copytree(CODE_DIR_DRIVE, CODE_DIR_LOCAL)
    print(f"Copied code from {CODE_DIR_DRIVE} to {CODE_DIR_LOCAL}")

config_path = CODE_DIR_LOCAL / "experiment_config.yaml"
with open(config_path) as f:
  cfg = yaml.safe_load(f)

  # Sincroniza rutas del config con las rutas del notebook
cfg['data']['xml_dir'] = str(DATA_DIR)
cfg['output']['base_dir'] = str(RESULTS_DIR)

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False, default_flow_style=False)

print("Experimento:", cfg['experiment']['name'])
print("Datos:     ", cfg['data']['xml_dir'])
print("Salidas:   ", cfg['output']['base_dir'])
print()
print("Fases activas:")
for k in ['phase_1_imputation', 'phase_2_variants', 'phase_3_efficiency', 'phase_4_stress', 'phase_5_multimodal']:
    print(f"  {k}: {cfg[k]['enabled']}")
print()
print(f"Variantes: {cfg['phase_2_variants']['variants']}")
print(f"Semillas:  {cfg['random']['seeds']}")

Copied code from /content/drive/MyDrive/data_tesis/code to /content/code
Experimento: tabular_conn_baseline_q1
Datos:      /content/drive/MyDrive/data_tesis/clinicalsupplement
Salidas:    /content/drive/MyDrive/data_tesis/results

Fases activas:
  phase_1_imputation: True
  phase_2_variants: True
  phase_3_efficiency: True
  phase_4_stress: True
  phase_5_multimodal: True

Variantes: ['cox_baseline', 'tabpfn', 'linear_fpga']
Semillas:  [42, 123, 456, 789, 1024]


## 5. (Opcional) Inspeccionar componentes registrados

In [6]:
import sys
sys.path.append(str(CODE_DIR_LOCAL))

from registry import list_components
import json
print(json.dumps(list_components(), indent=2))

{
  "imputation": [
    "knn_10",
    "knn_5",
    "mean_median",
    "mice"
  ],
  "variants": [
    "cox_baseline",
    "linear_fpga",
    "tabpfn"
  ],
  "text_conn": [
    "text_baseline_docling_clinicalbert"
  ],
  "vision_conn": [
    "vision_baseline_stunet_radiomics"
  ],
  "fusion_proc": [
    "fusion_baseline_concat"
  ],
  "prognosis_proc": [
    "prognosis_baseline_linear_cox"
  ]
}


## Nota sobre el baseline multimodal (Fase 5)

La Fase 5 corre el pipeline end-to-end completo: TABULAR-CONN + TEXT-CONN + VISION-CONN → FUSION-PROC → PROGNOSIS-PROC, y reporta C-index para varias combinaciones de modalidades (ablación).

**Comportamiento por defecto si no tienes datos de texto/visión:**
- Si `text_dir` y `vision_dir` son `null` en el config, esos conectores corren en MOCK mode (embeddings deterministas pero sintéticos). Esto valida la arquitectura pero los C-index multimodales no son significativos.
- En cuanto tengas reportes de patología (PDFs) y/o volúmenes CT (.nii.gz), apunta `text_dir` y `vision_dir` a esas carpetas en el config y vuelve a correr — sin tocar nada más.

## 6. Ejecutar el experimento

In [7]:
# Lee el config, computa el hash, crea el directorio del run, ejecuta todas las fases activas y deja todo en Drive con trazabilidad completa.
import os
os.chdir(CODE_DIR_LOCAL)
from experiment_runner import run_experiment
summary = run_experiment("experiment_config.yaml")

CLINICAL-CORE / TABULAR-CONN EXPERIMENT
Name:      tabular_conn_baseline_q1
Hash:      769367b4
Run dir:   /content/drive/MyDrive/data_tesis/results/20260409_004312_769367b4

[STEP 0] Extracting clinical data from XMLs
Found 621 XML files
  Successfully parsed: 621 cases

EXTRACTION QUALITY REPORT — 621 cases

Variable                             Present  Missing  %Complete
---------------------------------------------------------------
age                                      537       84      86.5%
gender                                   537       84      86.5%
race                                       0      621       0.0%
ethnicity                                385      236      62.0%
pathologic_stage                         554       67      89.2%
pathologic_T                             556       65      89.5%
pathologic_N                             267      354      43.0%
pathologic_M                             515      106      82.9%
histologic_grade                       

## 7. Inspeccionar el último run

In [8]:
# Encuentra el run más reciente
runs = sorted(RESULTS_DIR.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
if runs:
    latest = runs[0]
    print(f"Último run: {latest.name}")
    print()
    # Lista todos los archivos generados
    for f in sorted(latest.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<40} {size_kb:>8.1f} KB")
    # Muestra el summary
    import json
    with open(latest / "summary.json") as f:
        s = json.load(f)
    print()
    print("=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(json.dumps(s, indent=2, default=str))

Último run: 20260409_004312_769367b4

  experiment_config.yaml                        1.7 KB
  feature_config.yaml                           8.2 KB
  phase3_efficiency.csv                         0.2 KB
  phase5_modality_manifest.csv                 12.2 KB
  raw_features.csv                             49.3 KB
  raw_targets.csv                              12.5 KB
  run_metadata.json                             0.8 KB
  summary.json                                  1.0 KB

SUMMARY
{
  "phases": {
    "phase_3": [
      {
        "latency_ms": 0.3311,
        "latency_std_ms": 2.815,
        "memory_mb": 0.0,
        "flops": -1,
        "n_parameters": 0,
        "fpga_compatible": false,
        "variant": "cox_baseline"
      },
      {
        "latency_ms": 0.2083,
        "latency_std_ms": 0.0586,
        "memory_mb": 0.3955,
        "flops": 205696,
        "n_parameters": 103680,
        "fpga_compatible": true,
        "variant": "linear_fpga"
      }
    ]
  },
  "errors": [
 

## 8. Comparar resultados entre runs

In [9]:
import json
import pandas as pd

rows = []
for run_dir in sorted(RESULTS_DIR.iterdir()):
    summary_file = run_dir / "summary.json"
    if not summary_file.exists():
        continue
    with open(summary_file) as f:
        s = json.load(f)
        row = {
        'run': run_dir.name,
        'experiment': s.get('phases', {}).get('phase_1', {}).get('best_strategy', '?'),
        'n_cases': s.get('n_cases'),
        'runtime_s': s.get('runtime_seconds'),
        'errors': len(s.get('errors', [])),
    }
        # Extraer C-index promedio por variante (Fase 2)
    ph2 = s.get('phases', {}).get('phase_2', {})
    if ph2 and 'mean' in ph2:
        for variant, ci in ph2['mean'].items():
            row[f'ci_{variant}'] = ci
        rows.append(row)

if rows:
    df_runs = pd.DataFrame(rows)
    print(df_runs.to_string(index=False))
else:
    print("Aún no hay runs completados.")

Aún no hay runs completados.


## 9. (Opcional) Sincronizar el código local con Drive

In [10]:
SYNC_BACK_TO_DRIVE = False  # Cambiar a True para confirmar

if SYNC_BACK_TO_DRIVE:
  # Note: 'required_files' is not defined in the current notebook state.
  # If SYNC_BACK_TO_DRIVE is set to True, this loop will cause a NameError.
  # You would need to define 'required_files' before this block.
  # For demonstration, assuming it would be defined if this block were active.
  if 'required_files' in locals() or 'required_files' in globals():
    for fname in required_files:
      src = CODE_DIR_LOCAL / fname
      dst = CODE_DIR_DRIVE / fname
      if src.exists():
        shutil.copy2(src, dst)
        print(f"  ✓ Sync: {fname}")
  else:
    print("Advertencia: 'required_files' no está definido. No se puede realizar la sincronización.")
else:
  print("Sync deshabilitado. Cambia SYNC_BACK_TO_DRIVE=True para activarlo.")

Sync deshabilitado. Cambia SYNC_BACK_TO_DRIVE=True para activarlo.
